In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [ ]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
import numpy as np
import random

#generate 0,1, or 2 with equal probability
def randomThird():
    q = QuantumCircuit(1)
    op = np.array([[1/math.sqrt(3), math.sqrt(2)/math.sqrt(3)], [math.sqrt(2)/math.sqrt(3), -1/math.sqrt(3)]])
    q.unitary(op, qubits = 0, label = "A")

    q.measure_all() 
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    
    if counts.get("1", 0) == 1:
        return randomHalf()
    else:
        return 2

#generate 0 or 1 with equal probability
def randomHalf():
    q = QuantumCircuit(1) 
    q.h(0) 
    q.measure_all() 
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    return counts.get("1",0)

#create operators W and V from matrices
root2 = math.sqrt(2)
denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2) 

W_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]
W_transform = Operator(W_transform_matrix) 


V_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]
V_transform = Operator(V_transform_matrix) 

#define number of loops from N
N = 500
loops = (9*N)//2

aliceOps = ['X', 'W', 'Z']
bobOps = ['W', 'Z', 'V']

#initialise measurement, base lists for alice and bob
aliceMeasurements = [0]*loops
bobMeasurements = [0]*loops
aliceBases = [0]*loops
bobBases = [0]*loops

backend = BasicSimulator()

#loop for each qubit
for i in range(loops):
    q = QuantumCircuit(2) 
    q.h(0)
    q.cx(0, 1)
    
    aliceChoice = randomThird()
    bobChoice = randomThird()

    aliceBases[i] = aliceChoice
    bobBases[i] = bobChoice
    
    aliceOp = aliceOps[aliceChoice]
    bobOp = bobOps[bobChoice]

    #apply unitary operators based on choice
    match aliceOp:
        case 'X':
            q.h(0)
        case 'W':
            q.unitary(W_transform, [0], label = "W")
        case 'Z':
            pass

    match bobOp:
        case 'W':
            q.unitary(W_transform, [1], label = "W")
        case 'Z':
            pass
        case 'V':
            q.unitary(V_transform, [1], label = "V")

    
    q.measure_all()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)

    #list of the 2 bits in this loop, in order Bob, Alice
    bits = list(counts.keys())[0]

    #store whether bit is 0 or 1 for alice and bob respectively
    aliceMeasurements[i] = int(bits[1])
    bobMeasurements[i] = int(bits[0])

#initialise shared key lists
sharedKey = []
xw = []
xv = []
zw = []
zv = []

#loop through bases to add bits to shared key when valid
for x in range(len(aliceBases)):
    i = aliceBases[x]
    j = bobBases[x]

    #if choices line up, add to key
    if (i == 1 and j == 0) or (i == 2 and j == 1):
        sharedKey.append(aliceMeasurements[x])
    #add to combination lists to calculate s
    else:
        if i == 0 and j == 0:
            aliceQ = 1 if aliceMeasurements[x] == 0 else -1
            bobQ = 1 if bobMeasurements[x] == 0 else -1
            xw.append(aliceQ * bobQ)
            
        elif i == 0 and j == 2:
            aliceQ = 1 if aliceMeasurements[x] == 0 else -1
            bobQ = 1 if bobMeasurements[x] == 0 else -1
            xv.append(aliceQ * bobQ)
            
        elif i == 2 and j == 0:
            aliceQ = 1 if aliceMeasurements[x] == 0 else -1
            bobQ = 1 if bobMeasurements[x] == 0 else -1
            zw.append(aliceQ * bobQ)
            
        elif i == 2 and j == 2:
            aliceQ = 1 if aliceMeasurements[x] == 0 else -1
            bobQ = 1 if bobMeasurements[x] == 0 else -1
            zv.append(aliceQ * bobQ)

#calculate s with combination lists
s = abs(np.mean(xw) - np.mean(xv) + np.mean(zw) + np.mean(zv))

print(s)
print(sharedKey)